# 02: Retention, RFM and delivery impact

Input: data/processed/orders_enriched.csv (one row per order, built in notebook 01).
Rules: customers are identified by customer_unique_id; canceled and unavailable orders are excluded;
the cohort window is 2017-01 to 2018-08 (2016 and the last two months of 2018 are incomplete).

In [1]:
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv

repo = Path(find_dotenv(usecwd=True)).parent
date_cols = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
             "order_delivered_customer_date", "order_estimated_delivery_date"]
orders = pd.read_csv(repo / "data" / "processed" / "orders_enriched.csv", parse_dates=date_cols)

print("shape:", orders.shape)
print("one row per order:", orders["order_id"].is_unique)
print("orders with order_number:", orders["order_number"].notna().sum())
print("purchase column type:", orders["order_purchase_timestamp"].dtype)

shape: (99441, 18)
one row per order: True
orders with order_number: 98207
purchase column type: datetime64[ns]


**Check:** orders_enriched.csv loads with 99,441 orders and 18 columns, one row per order, and the
dates parse as datetimes. 98,207 orders have an order_number (canceled and unavailable orders are
excluded), the same count as SQL Query 13.

In [2]:
first = orders[orders["order_number"] == 1].copy()
first["cohort_month"] = first["order_purchase_timestamp"].dt.to_period("M")
cohort_size = first.groupby("cohort_month").size()

print("people with a first order:", len(first))
print("people who ordered again:", (first["person_total_orders"] > 1).sum())
print(cohort_size)

people with a first order: 94990
people who ordered again: 2888
cohort_month
2016-09       2
2016-10     290
2016-12       1
2017-01     752
2017-02    1690
2017-03    2571
2017-04    2325
2017-05    3541
2017-06    3102
2017-07    3822
2017-08    4130
2017-09    4075
2017-10    4392
2017-11    7190
2017-12    5439
2018-01    6951
2018-02    6357
2018-03    6931
2018-04    6698
2018-05    6586
2018-06    5920
2018-07    6016
2018-08    6209
Freq: M, dtype: int64


**Insight:** 94,990 people have a first order, and 2,888 of them (3.04%) ordered again, matching SQL
Query 13. The 2016 cohorts are tiny (2, 290 and 1 people, November missing), so the usable window is
2017-01 to 2018-08: 20 cohorts and 94,697 people. The largest cohort is 2017-11 (7,190 people), and the
2018 cohorts stay between 5,920 and 6,951, in line with the flat revenue seen earlier. There is no
cohort for 2018-09 or 2018-10, because nobody's first valid order falls in those months (cause not checked).

In [3]:
valid = orders[orders["order_number"].notna()].copy()
valid["order_month"] = valid["order_purchase_timestamp"].dt.to_period("M")
valid = valid.merge(first[["customer_unique_id", "cohort_month"]], on="customer_unique_id", how="left")

valid["months_since_first"] = ((valid["order_month"].dt.year - valid["cohort_month"].dt.year) * 12
                               + (valid["order_month"].dt.month - valid["cohort_month"].dt.month))

active = (valid.groupby(["cohort_month", "months_since_first"])["customer_unique_id"].nunique()
          .unstack(fill_value=0))

print("rows:", len(valid))
print("repeat orders (order_number > 1):", (valid["order_number"] > 1).sum())
print("repeat orders in the same month as the first order:",
      ((valid["order_number"] > 1) & (valid["months_since_first"] == 0)).sum())
print("month 0 equals cohort size for every cohort:", (active[0] == cohort_size).all())

rows: 98207
repeat orders (order_number > 1): 3217
repeat orders in the same month as the first order: 1273
month 0 equals cohort size for every cohort: True


**Insight:** 98,207 orders belong to 94,990 people, which leaves 3,217 repeat orders, about 1.11 per
repeat buyer (2,888). Month 0 equals the cohort size for every cohort, so the counting is consistent.
1,273 repeat orders (39.6%) happen in the same calendar month as the first order. Caution: the matrix
counts people active in each month, so someone who bought again in the first month is not visible in
month 1 or later, and the later columns understate how many people ever returned.